# 🚗 Car GAN – Training Notebook
Run this notebook on **Google Colab** (T4 GPU recommended).
Runtime → Change runtime type → GPU

In [ ]:
# ── 1. Check GPU ────────────────────────────────────────────
!nvidia-smi

In [ ]:
# ── 2. Clone repo ───────────────────────────────────────────
!git clone https://github.com/YOUR_USERNAME/car-gan.git
%cd car-gan

In [ ]:
# ── 3. Install dependencies ─────────────────────────────────
!pip install -q torch torchvision hydra-core omegaconf kaggle huggingface_hub

In [ ]:
# ── 4. Upload kaggle.json to download dataset ────────────────
# Skip if you have the data already
from google.colab import files
files.upload()   # upload your kaggle.json here

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
# ── 5. Download Stanford Cars dataset ───────────────────────
from src.data.dataset import download_stanford_cars
download_stanford_cars('./data')

In [ ]:
# ── 6. Quick sanity check on dataloader ─────────────────────
from src.data.dataset import build_dataloader
import matplotlib.pyplot as plt
import numpy as np

loader = build_dataloader('./data/car_data/car_data/train', image_size=64, batch_size=16, num_workers=2)
imgs, _ = next(iter(loader))

grid = ((imgs[:8] + 1) / 2).permute(0, 2, 3, 1).numpy()
fig, axes = plt.subplots(1, 8, figsize=(16, 2))
for ax, img in zip(axes, grid):
    ax.imshow(img)
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# ── 7. Mount Google Drive for persistent checkpoints ────────
from google.colab import drive
drive.mount('/content/drive')

import yaml

# Override checkpoint dir in config
with open('configs/config.yaml') as f:
    cfg = yaml.safe_load(f)

cfg['logging']['checkpoint_dir'] = '/content/drive/MyDrive/car-gan/checkpoints'
cfg['logging']['log_dir']        = '/content/drive/MyDrive/car-gan/runs'

with open('configs/config.yaml', 'w') as f:
    yaml.dump(cfg, f)

print('Checkpoints will be saved to Google Drive.')

In [ ]:
# ── 8. Train! ────────────────────────────────────────────────
# Runs train.py via hydra; override epochs inline
!python train.py training.epochs=200 data.dataset_path=./data/car_data/car_data/train

In [ ]:
# ── 9. Visualise loss curves ─────────────────────────────────
# (runs after training or load history from json)
from src.utils.visualization import plot_loss_curves
import json, glob

# Just demo with dummy data – replace with real history
history = [{'epoch': i, 'g_loss': 2.5 - i*0.01, 'd_loss': 1.2 + i*0.002} for i in range(1, 11)]
plot_loss_curves(history)

In [ ]:
# ── 10. Push generator weights to HuggingFace Hub ───────────
import torch
from huggingface_hub import HfApi
from src.models import Generator
from src.utils.checkpoint import load_checkpoint

HF_TOKEN = 'hf_...'      # ← your HF write token
REPO_ID  = 'your-username/car-gan'

# Load best checkpoint
G = Generator(latent_dim=128, features=64, channels=3, image_size=64)
D_dummy = __import__('src.models', fromlist=['Discriminator']).Discriminator()
opt_g = torch.optim.Adam(G.parameters())
opt_d = torch.optim.Adam(D_dummy.parameters())
load_checkpoint('checkpoints/ckpt_final.pth', G, D_dummy, opt_g, opt_d)

# Save generator-only weights
torch.save({'generator_state': G.state_dict()}, 'generator_weights.pth')

api = HfApi()
api.create_repo(repo_id=REPO_ID, exist_ok=True, token=HF_TOKEN)
api.upload_file(
    path_or_fileobj='generator_weights.pth',
    path_in_repo='generator_weights.pth',
    repo_id=REPO_ID,
    token=HF_TOKEN,
)
print(f'Uploaded to https://huggingface.co/{REPO_ID}')